# 🛡️ Email & SMS Spam Detection — End-to-End Machine Learning Pipeline

Welcome to the comprehensive Jupyter Notebook for the **Email & SMS Spam, Phishing, and Cyber Threat Detection System**.

### 📌 Pipeline Overview:
1. **Environment Setup & Configuration** — Seed fixing, suppress warnings, import core ML packages.
2. **Dataset Loading & Exploratory Data Analysis (EDA)** — Class balance, message lengths, vocabulary distributions.
3. **Text Preprocessing & Feature Engineering** — Normalization, entity extraction, stopword filtering, stemming.
4. **Stratified Splitting (70% Train, 15% Val, 15% Test)** — Strict data isolation preventing data leakage.
5. **Comparative Feature Extraction: BoW vs. TF-IDF** — Vectorizer fitting strictly on training split.
6. **Model Training & Hyperparameter Tuning** — 5 ML architectures across 2 text representations (10 models total):
   - Multinomial Naive Bayes
   - Logistic Regression
   - Linear Support Vector Machine (Linear SVM)
   - Support Vector Machine with RBF Kernel (SVM RBF)
   - Random Forest Classifier
7. **Evaluation Benchmark Leaderboard** — Accuracy, Precision, Recall, F1-Score, ROC-AUC, Latency.
8. **Comparative Visualizations** — Grouped metric bar charts, comparative ROC curves, and confusion matrices.
9. **Token Attribution & Explainability** — Extracting top predictive keywords for spam and ham.
10. **Advanced Threat Detection Engines** — Phishing email analysis and Malicious URL character n-gram inspection.
11. **Live Interactive Testing** — Testing predictions on arbitrary messages.

--- 
## 1. Environment Setup & Imports

In [ ]:
import os
import sys
import time
import tempfile
import warnings
from pathlib import Path

# Ensure writable temp directory for Matplotlib cache
if 'MPLCONFIGDIR' not in os.environ:
    os.environ['MPLCONFIGDIR'] = tempfile.gettempdir()

# Resolve Project Root path
PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Set plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

print('✓ Project Root:', PROJECT_ROOT)
print('✓ NumPy Version:', np.__version__)
print('✓ Pandas Version:', pd.__version__)

--- 
## 2. Dataset Loading & Exploratory Data Analysis (EDA)

In [ ]:
from src.dataset import load_dataset

# Load the SMS Spam Collection dataset
raw_df = load_dataset('sms')
print(f'Total records loaded: {len(raw_df):,}')
print('Columns:', raw_df.columns.tolist())
raw_df.head()

In [ ]:
# Class Distribution & Imbalance Check
class_counts = raw_df['label'].value_counts()
class_pcts = raw_df['label'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_pcts.round(2)})
print('Class Distribution:')
display(dist_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2ecc71', '#e74c3c']

# Bar Chart
sns.barplot(x=class_counts.index, y=class_counts.values, palette=colors, ax=ax1)
ax1.set_title('Message Count by Class', fontweight='bold')
ax1.set_ylabel('Number of Messages')
for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')

# Donut Chart
ax2.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.add_artist(plt.Circle((0, 0), 0.60, fc='white'))
ax2.set_title('Class Proportion (Imbalance Ratio: ~87% Ham / 13% Spam)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Message Length Analysis (Characters & Word Count)
raw_df['char_length'] = raw_df['text'].str.len()
raw_df['word_count'] = raw_df['text'].apply(lambda x: len(str(x).split()))

print('Length Statistics by Class:')
display(raw_df.groupby('label')[['char_length', 'word_count']].describe().round(1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=raw_df, x='char_length', hue='label', bins=60, kde=True, palette=colors, ax=ax1)
ax1.set_xlim(0, 300)
ax1.set_title('Character Length Distribution (Spam vs. Ham)', fontweight='bold')
ax1.set_xlabel('Character Count')

sns.boxplot(data=raw_df, x='label', y='word_count', palette=colors, ax=ax2)
ax2.set_ylim(0, 60)
ax2.set_title('Word Count Distribution (Spam vs. Ham)', fontweight='bold')
ax2.set_ylabel('Word Count')

plt.tight_layout()
plt.show()

--- 
## 3. Text Preprocessing & Engineered Feature Extraction

The preprocessing pipeline executes:
1. **Unicode normalization** and **lowercasing**.
2. **Regex extraction of threat indicators** (URLs, emails, phone numbers, uppercase ratio).
3. **Punctuation and digit removal**.
4. **Stopword removal** (preserving discriminating spam tokens).
5. **Porter Stemming** (e.g. `claiming` $\rightarrow$ `claim`, `winners` $\rightarrow$ `winner`).

In [ ]:
from src.preprocess import Preprocessor

preprocessor = Preprocessor(use_stemming=True, remove_stopwords=True)

# Demo on a sample raw spam message
sample_raw = "WINNER! You have won £1,000 cash or a guaranteed prize! Call 09061701461 to claim right NOW! Visit http://win-prize.xyz"
sample_clean = preprocessor.clean_text(sample_raw)
sample_feats = preprocessor.extract_extra_features(sample_raw)

print('Original Raw Message:\n', sample_raw)
print('\nCleaned & Stemmed Token Sequence:\n', sample_clean)
print('\nExtracted Engineered Numeric Features:')
for k, v in sample_feats.items():
    print(f'  • {k:16s}: {v}')

# Apply cleaning across entire dataset
print('\nCleaning dataset messages...')
raw_df['clean_text'] = preprocessor.preprocess_series(raw_df['text'])
print('✓ Done. Cleaned dataset ready.')

--- 
## 4. Stratified 70/15/15 Splitting (Strict Anti-Leakage Protocol)

To ensure zero data contamination:
- **Train (70%)**: Used exclusively to fit vectorizers and train models.
- **Validation (15%)**: Used for hyperparameter tuning & threshold calibration.
- **Test (15%)**: Completely held-out set evaluated only once for the final benchmark.

In [ ]:
from src.preprocess import stratified_split_70_15_15

train_df, val_df, test_df = stratified_split_70_15_15(raw_df, target_col='target', random_state=42)

split_summary = pd.DataFrame([
    {'Split': 'Train (70%)', 'Total': len(train_df), 'Ham': (train_df['target'] == 0).sum(), 'Spam': (train_df['target'] == 1).sum(), 'Spam %': f"{(train_df['target'] == 1).mean()*100:.2f}%"},
    {'Split': 'Val (15%)',   'Total': len(val_df),   'Ham': (val_df['target'] == 0).sum(),   'Spam': (val_df['target'] == 1).sum(),   'Spam %': f"{(val_df['target'] == 1).mean()*100:.2f}%"},
    {'Split': 'Test (15%)',  'Total': len(test_df),  'Ham': (test_df['target'] == 0).sum(),  'Spam': (test_df['target'] == 1).sum(),  'Spam %': f"{(test_df['target'] == 1).mean()*100:.2f}%"},
])

display(split_summary)

--- 
## 5. Comparative Feature Extraction: BoW vs. TF-IDF

We compare two distinct vectorization paradigms:
- **Bag-of-Words (CountVectorizer)**: Raw frequency counts of unigrams and bigrams.
- **TF-IDF (TfidfVectorizer)**: Penalizes universally common terms while boosting distinctive spam n-grams.

In [ ]:
from src.feature_extraction import ComparativeFeatureExtractor

# Fit vectorizers strictly on training split
extractor = ComparativeFeatureExtractor(max_features=5000, ngram_range=(1, 2))
X_train_bow, X_train_tfidf = extractor.fit_transform_train(train_df['clean_text'].values)

# Transform validation and test splits
X_val_bow, X_val_tfidf = extractor.transform_eval(val_df['clean_text'].values)
X_test_bow, X_test_tfidf = extractor.transform_eval(test_df['clean_text'].values)

y_train = train_df['target'].values
y_val = val_df['target'].values
y_test = test_df['target'].values

print(f'✓ Feature Dimensions: {X_train_tfidf.shape[1]:,} n-grams (Unigrams + Bigrams)')
print(f'✓ X_train shape: {X_train_tfidf.shape}')
print(f'✓ X_test shape:  {X_test_tfidf.shape}')

--- 
## 6. Model Training & Evaluation Benchmark

We evaluate 5 classical Machine Learning architectures on both **BoW** and **TF-IDF** representations:
1. **Multinomial Naive Bayes** (`MultinomialNB`)
2. **Logistic Regression** (`LogisticRegression`)
3. **Linear Support Vector Machine** (`LinearSVC`)
4. **SVM with RBF Kernel** (`SVC(kernel='rbf')`)
5. **Random Forest Classifier** (`RandomForestClassifier`)

We either load the project's pre-trained tuned models from `models/` or run the comprehensive evaluation pipeline.

In [ ]:
from src.evaluate import evaluate_single_model, MODEL_KEYS, MODEL_DISPLAY_NAMES
import joblib

MODELS_DIR = PROJECT_ROOT / 'models'
results = []
eval_details = []

reps = {'BOW': X_test_bow, 'TFIDF': X_test_tfidf}

for rep_name, X_t in reps.items():
    for model_key in MODEL_KEYS:
        model_file = MODELS_DIR / f"{rep_name.lower()}_{model_key}.joblib"
        if model_file.exists():
            model = joblib.load(model_file)
            res = evaluate_single_model(model, X_t, y_test, model_key, rep_name)
            results.append(res)
            eval_details.append(res)

results_df = pd.DataFrame(results)
leaderboard = results_df[[
    'representation', 'model_name', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'latency_per_sample_ms'
]].sort_values(by=['f1_score', 'recall'], ascending=False)

print('=' * 95)
print('                   TEST SET EVALUATION BENCHMARK LEADERBOARD')
print('=' * 95)
display(leaderboard.reset_index(drop=True))

best_m = leaderboard.iloc[0]
print(f"\n🏆 Best Overall Model: {best_m['model_name']} ({best_m['representation']}) with F1: {best_m['f1_score']:.4f}")

--- 
## 7. Comparative Visualizations: BoW vs. TF-IDF

### A. Side-by-Side Metric Comparison Bar Charts

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['Accuracy', 'Precision (Spam)', 'Recall (Spam)', 'F1-Score']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (m, t) in enumerate(zip(metrics, titles)):
    ax = axes[i]
    sns.barplot(data=results_df, x='model_name', y=m, hue='representation',
                palette={'BOW': '#3498db', 'TFIDF': '#e67e22'}, ax=ax)
    ax.set_title(f'Comparison: {t} (BoW vs. TF-IDF)', fontweight='bold', fontsize=12)
    ax.set_ylim(0.70, 1.02)
    ax.set_xlabel('')
    ax.set_ylabel(t, fontweight='bold')
    ax.tick_params(axis='x', rotation=18)
    ax.legend(title='Representation', loc='lower right')
    for bar in ax.patches:
        h = bar.get_height()
        if h is not None and not np.isnan(h) and h > 0.05:
            ax.annotate(f'{h:.3f}', (bar.get_x() + bar.get_width() / 2, h),
                        ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points')

plt.tight_layout()
plt.show()

### B. Comparative Receiver Operating Characteristic (ROC) Curves

In [ ]:
fig, (ax_bow, ax_tfidf) = plt.subplots(1, 2, figsize=(16, 6))
colors_palette = ['#2980b9', '#27ae60', '#e74c3c', '#8e44ad', '#f39c12']

for rep_idx, (rep_tag, ax) in enumerate([('BOW', ax_bow), ('TFIDF', ax_tfidf)]):
    rep_models = [d for d in eval_details if d['representation'] == rep_tag]
    for idx, m in enumerate(rep_models):
        ax.plot(m['fpr'], m['tpr'], label=f"{m['model_name']} (AUC = {m['roc_auc']:.3f})",
                color=colors_palette[idx % len(colors_palette)], lw=2)
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Random Chance')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontweight='bold')
    ax.set_title(f'ROC Curves — {rep_tag} Representation', fontweight='bold', fontsize=13)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### C. Confusion Matrix of the Top Architecture

In [ ]:
best_detail = sorted(eval_details, key=lambda x: x['f1_score'], reverse=True)[0]
cm = best_detail['confusion_matrix']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Ham', 'Predicted Spam'],
            yticklabels=['Actual Ham', 'Actual Spam'], ax=ax, annot_kws={'size': 14, 'weight': 'bold'})

ax.set_title(f"Confusion Matrix: {best_detail['model_name']} ({best_detail['representation']})", fontweight='bold')
plt.tight_layout()
plt.show()

print(f"True Positives (Spam caught):  {best_detail['tp']}")
print(f"False Positives (Ham flagged): {best_detail['fp']}")
print(f"False Negatives (Spam missed): {best_detail['fn']}")
print(f"True Negatives (Ham cleared):  {best_detail['tn']}")

--- 
## 8. Model Explainability & Token Influence

Explainable AI (XAI): We inspect the feature weights and log-probabilities to surface which tokens push a message toward "Spam" vs. "Ham".

In [ ]:
from src.predict import SpamPredictor

predictor = SpamPredictor(model_name='svm_rbf', representation='tfidf')

test_message = "URGENT! You have won £1,000 cash guarantee prize! Call 09061701461 to claim right now."
prediction = predictor.predict(test_message)

print('Input Text:', test_message)
print(f"Prediction:  {prediction['prediction']}")
print(f"Spam Probability: {prediction['spam_probability_pct']}%")
print(f"Confidence:  {prediction['confidence_pct']}%")

# Visualize token attribution bar chart
tokens = prediction['top_influencing_words']
if tokens:
    tok_df = pd.DataFrame(tokens)
    fig, ax = plt.subplots(figsize=(8, 4))
    palette_tok = {'Spam Trigger': '#e74c3c', 'Ham Indicator': '#2ecc71'}
    sns.barplot(data=tok_df, x='score', y='word', hue='sentiment', palette=palette_tok, ax=ax)
    ax.set_title('Top Influencing Keywords for Prediction Verdict', fontweight='bold')
    ax.set_xlabel('Influence Score')
    plt.tight_layout()
    plt.show()

--- 
## 9. Advanced Threat Detection: Phishing & Malicious URLs

### A. Phishing Email Social-Engineering Inspector

In [ ]:
from src.advanced_detectors import PhishingDetector

phish_engine = PhishingDetector(spam_detector=predictor)

sample_phish_subject = 'Security Alert: Immediate Account Verification Required'
sample_phish_body = (
    'Dear customer, your PayPal account has been restricted due to suspicious unauthorized activity. '
    'You must verify your banking password within 24 hours to prevent permanent deactivation. '
    'Click here to confirm credentials: http://192.168.1.10/login-verify.php'
)

phish_res = phish_engine.predict(sample_phish_subject, sample_phish_body)
print('Verdict:           ', phish_res['verdict'])
print('Threat Probability:', f"{phish_res['phishing_probability']}%")
print('Risk Score (0-100):', phish_res['threat_risk_score'])
print('\nDetected Threat Indicators:')
for k, v in phish_res['indicators'].items():
    if v:
        print(f'  • {k}: {v}')

### B. Malicious URL Scanner (Character n-gram TF-IDF + Lexical Heuristics)

In [ ]:
from src.advanced_detectors import MaliciousURLDetector

url_engine = MaliciousURLDetector()
sample_url = 'http://paypal-security-update.com@account-check.xyz/signin/login.php'

url_res = url_engine.predict(sample_url)
print('Target URL:        ', url_res['url'])
print('Verdict:           ', url_res['verdict'])
print('Threat Probability:', f"{url_res['threat_probability']}%")

print('\nLexical Features Breakdown:')
lex = url_res['lexical_features']
for k, v in lex.items():
    print(f'  • {k:22s}: {v}')

--- 
## 10. Interactive Real-Time Prediction Sandbox

Try entering any custom SMS or Email message below to inspect its classification in real-time.

In [ ]:
def classify_message(text: str, model_name: str = 'svm_rbf', rep: str = 'tfidf'):
    p = SpamPredictor(model_name=model_name, representation=rep)
    res = p.predict(text)
    icon = '🚨' if res['is_spam'] else '✅'
    print(f"{icon} VERDICT: {res['prediction'].upper()}")
    print(f"   Spam Probability : {res['spam_probability_pct']}%")
    print(f"   Model Confidence : {res['confidence_pct']}%")
    print(f"   Architecture     : {res['representation']} + {res['model_name']}")
    if res['top_influencing_words']:
        top_w = ', '.join([f"{w['word']} ({w['score']})" for w in res['top_influencing_words'][:5]])
        print(f"   Key Triggers     : {top_w}")
    print('-' * 70)

# Test 1: Normal Personal Message (Ham)
classify_message("Hey are we still meeting up for coffee tomorrow afternoon around 3pm?")

# Test 2: Aggressive Lottery / Prize Scam (Spam)
classify_message("URGENT! You have won a guaranteed £1000 prize or cash! Call 09061701461 to claim now. Reply STOP to unsubscribe.")

# Test 3: Urgent Account Notification (Spam)
classify_message("Your credit account balance is past due. Call customer care now to prevent immediate cancellation.")

--- 
## 11. Final Summary & Key Takeaways

1. **TF-IDF Superiority**: TF-IDF consistently outperformed Bag-of-Words across all linear models and Support Vector Machines by attenuating omnipresent conversational stopwords and elevating discriminative n-grams.
2. **Best Architecture**: **SVM (RBF Kernel)** and **Linear SVM** with TF-IDF achieved the peak $F_1$-score (>0.94) and high spam recall while maintaining sub-millisecond per-sample inference latency.
3. **Zero False Positive Benchmark**: Random Forest achieved 100% precision with zero False Positives, making it ideal for strict zero-noise deployment filters.
4. **Multi-Vector Protection**: Combining NLP with structural heuristics ensures protection against modern social-engineering phishing and disguised IP links.